# 01 · Exploratory Data Analysis: Home Credit Default Risk

**Project:** P1 Credit Risk Service · **Stage 3 (EDA)**
**Data:** Kaggle *Home Credit Default Risk*, stored in `data/raw/` (git-ignored, never redistributed).

Each section answers one question, and its answer decides something in a later stage.

> **Licence rule:** this notebook shows aggregates and plots only, never raw rows (no `head()` or `sample()`).

| # | Section | Decides | Status |
|---|---|---|---|
| 0 | Setup | — | ✅ |
| 1 | Shape and target | Cost-model default rate, stratification, metrics | ✅ |
| 2 | Data quality | Imputation, missing-indicator features, drops | ⏳ |
| 3 | Signal | Stage 5 feature-engineering priorities | ⏳ |
| 4 | Redundancy | Feature pruning for the logistic regression baseline | ⏳ |
| 5 | Fairness groundwork | Base rates for the Stage 6 fairness audit | ⏳ |
| 6 | Bureau history | Bureau aggregation plan, leakage check | ⏳ |
| 7 | Findings | Summary for the README and Stages 4–6 | ⏳ |

---
## 0. Setup

Imports, display options and paths. Paths are built from the repo root with `pathlib`, so the notebook runs on any machine as long as it sits in `notebooks/`. The printed pandas version matters: pandas 3.x loads text as the `str` dtype, pandas 2.x as `object`.

In [17]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)

print(f"pandas {pd.__version__} | numpy {np.__version__}")

pandas 3.1.1 | numpy 2.5.5


In [18]:
# The notebook runs from notebooks/, so the repo root is one level up.
ROOT = Path.cwd().parent
RAW_DIR = ROOT / "data" / "raw"
TRAIN_PATH = RAW_DIR / "application_train.csv"
TEST_PATH = RAW_DIR / "application_test.csv"

assert RAW_DIR.exists(), f"Raw data folder not found: {RAW_DIR}"

for path in sorted(RAW_DIR.iterdir()):
    print(f"{path.name:<40} {path.stat().st_size / 1024**2:>8.1f} MB")

application_test.csv                         25.3 MB
application_train.csv                       158.4 MB
bureau.csv                                  162.1 MB
bureau_balance.csv                          358.2 MB
credit_card_balance.csv                     404.9 MB
HomeCredit_columns_description.csv            0.0 MB
installments_payments.csv                   689.6 MB
POS_CASH_balance.csv                        374.5 MB
previous_application.csv                    386.2 MB
sample_submission.csv                         0.5 MB


---
## 1. Shape and target

**Question:** How big is the main table, is there exactly one row per applicant, and what is the default rate?

**Decides:** whether the 8% default rate assumed in the Stage 1 cost model holds, how the data is split (stratification), and which metrics are meaningful.

### 1.1 Peek at the column types

Read only the first 1,000 rows at default types to see which dtypes pandas infers, without loading the full 166 MB file.

In [19]:
sample_df = pd.read_csv(TRAIN_PATH, nrows=1_000)
sample_df.dtypes.value_counts()

float64    65
int64      41
str        16
Name: count, dtype: int64

**Result:** 65 `float64`, 41 `int64` and 16 text columns. On pandas 3.x the text columns show as `str` (the dedicated string dtype) rather than the older `object`.

### 1.2 Choose a dtype for each column

| Inferred dtype | Loaded as | Why |
|---|---|---|
| `int64` | `int32` | Halves memory; the value range is checked after loading in 1.3. |
| `float64` | `float32` | About 7 significant digits, enough for EDA and model inputs; precision is checked in 1.3. |
| text | `category` | Few unique values per column, so integer codes plus a lookup table save memory and speed up `groupby`. |

**Risk:** the rule comes from 1,000 rows. If an integer column had a missing value further down the file, `read_csv` would raise a `ValueError`, because an `int32` column cannot hold NaN. A clean full load in 1.3 is the evidence that this did not happen.

Text columns are selected with `exclude="number"`, which works on pandas 2 and 3 without a deprecation warning.

In [20]:
int_cols = sample_df.select_dtypes(include="int64").columns
float_cols = sample_df.select_dtypes(include="float64").columns
text_cols = sample_df.select_dtypes(exclude="number").columns

dtype_map = {
    **{col: "int32" for col in int_cols},
    **{col: "float32" for col in float_cols},
    **{col: "category" for col in text_cols},
}

# Every column must be covered exactly once.
assert len(dtype_map) == sample_df.shape[1], "dtype_map does not cover every column"
print(f"{len(int_cols)} int -> int32 | {len(float_cols)} float -> float32 | {len(text_cols)} text -> category")

41 int -> int32 | 65 float -> float32 | 16 text -> category


### 1.3 Load the full file and measure the saving

Load once at default types only to measure memory, free it, then load with `dtype_map`. `memory_usage(deep=True)` counts the real size of text values, not just pointers to them.

In [21]:
def memory_mb(df: pd.DataFrame) -> float:
    """Return a DataFrame's in-memory size in megabytes (deep, so text is counted)."""
    return df.memory_usage(deep=True).sum() / 1024**2


default_df = pd.read_csv(TRAIN_PATH)
mem_default_mb = memory_mb(default_df)
del default_df  # free the default-typed copy before the typed load

train_df = pd.read_csv(TRAIN_PATH, dtype=dtype_map)
mem_typed_mb = memory_mb(train_df)

print(f"Default dtypes: {mem_default_mb:,.1f} MB")
print(f"Typed load:     {mem_typed_mb:,.1f} MB")
print(f"Reduction:      {1 - mem_typed_mb / mem_default_mb:.0%}")

Default dtypes: 325.2 MB
Typed load:     129.0 MB
Reduction:      60%


In [22]:
# Check 1: every integer value fits comfortably in int32 (about ±2.1 billion).
int_values = train_df.select_dtypes(include="int32")
print(f"Integer range: {int_values.min().min():,} to {int_values.max().max():,}")

# Check 2: float32 precision on the money columns, against a float64 reference.
money_cols = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]
money_f64 = pd.read_csv(TRAIN_PATH, usecols=money_cols)
abs_error = (money_f64 - train_df[money_cols].astype("float64")).abs()

pd.DataFrame({
    "max_value": money_f64.max(),
    "rows_above_2^24": (money_f64 > 2**24).sum(),
    "max_abs_error": abs_error.max(),
})

Integer range: -25,229 to 456,255


,max_value,rows_above_2^24,max_abs_error
AMT_INCOME_TOTAL,117000000.0,2,0.0125
AMT_CREDIT,4050000.0,0,0.0000
AMT_ANNUITY,258025.5,0,0.0000
AMT_GOODS_PRICE,4050000.0,0,0.0000


**Result (pandas 3.1.1):** 325 MB → 129 MB, about 60% less memory. The default figure depends on the pandas version: pandas 2.x stores text as Python `object` strings (about 537 MB in total), while pandas 3.x uses the more compact `str` dtype.
- The load ran without errors, so no integer column has missing values in the full file.
- Integers range from −25,229 to 456,255, well inside `int32`.
- float32 stores whole numbers exactly only up to 2²⁴ = 16,777,216. Only 2 incomes exceed it, and the worst rounding error on any money column is 0.0125, which is negligible.

### 1.4 Structure

In [23]:
print(f"Rows: {train_df.shape[0]:,} | Columns: {train_df.shape[1]}")
print(f"One row per applicant (SK_ID_CURR unique): {train_df['SK_ID_CURR'].is_unique}")

# astype(str): each category column has its own CategoricalDtype, so without it
# value_counts() would list all 16 category columns separately.
train_df.dtypes.astype(str).value_counts()

Rows: 307,511 | Columns: 122
One row per applicant (SK_ID_CURR unique): True


float32     65
int32       41
category    16
Name: count, dtype: int64

**Result:** 307,511 rows × 122 columns (120 features, the ID `SK_ID_CURR` and `TARGET`). `SK_ID_CURR` is unique, so each row is one application, and the history tables must be aggregated *to* this key before joining.

### 1.5 Target and class balance

In [24]:
target_counts = train_df["TARGET"].value_counts().sort_index()
default_rate = train_df["TARGET"].mean()
imbalance_ratio = target_counts[0] / target_counts[1]

print(f"Default rate:    {default_rate:.2%}")
print(f"Imbalance ratio: {imbalance_ratio:.2f} non-defaults per default")

pd.DataFrame({
    "count": target_counts,
    "share_%": (target_counts / len(train_df) * 100).round(2),
}).rename(index={0: "repaid (0)", 1: "default (1)"})

Default rate:    8.07%
Imbalance ratio: 11.39 non-defaults per default


,count,share_%
TARGET,,
repaid (0),282686,91.93
default (1),24825,8.07


**Result:** 24,825 defaults out of 307,511 applications: a **default rate of 8.07%** and **11.39 non-defaults per default**.

### 1.6 Train vs test columns

`application_test.csv` has no `TARGET`, so it cannot be our test set: the hold-out is carved from `application_train` in Stage 4. Here we only check that its columns match what the model will receive at prediction time. `nrows=0` reads the header only.

In [25]:
test_cols = pd.read_csv(TEST_PATH, nrows=0).columns

print("In train, not in test:", sorted(set(train_df.columns) - set(test_cols)))
print("In test, not in train:", sorted(set(test_cols) - set(train_df.columns)))

In train, not in test: ['TARGET']
In test, not in train: []


**Result:** only `TARGET` differs. Every feature available at training time is also present at prediction time.

### 1.7 Findings and implications

1. **The default rate holds.** Measured 8.07% against the 8% assumed in Stage 1. Approve-all floor: 125,000 − 475,000 × 0.0807 ≈ **₦86,650 per application** (was ₦87,000), a negligible change, so the cost model stands. `TARGET` means payment difficulty, not outright default, so C_FN remains an upper bound.
2. **Accuracy is useless here.** Predicting "repaid" for everyone scores 91.9% accuracy and catches no defaulters. Models are compared on PR-AUC (random baseline ≈ 0.081, the default rate, not 0.5), recall at the chosen threshold, and expected profit at the cost threshold p* ≈ 0.26. ROC-AUC is reported too, but it can look strong while precision at a usable threshold is poor.
3. **Stratify every split.** The hold-out and each CV fold keep about 8.07% defaults. A 20% hold-out still contains about 4,965 defaults, enough for stable estimates.
4. **The imbalance is moderate (11.4 : 1).** Try class weights first. SMOTE is an experiment only, fitted inside CV folds through an imblearn Pipeline.
5. **float32 is safe for money columns in EDA and as model inputs** (see 1.3). Profit and cost calculations, and large sums, use float64.
6. **`category` is for EDA only.** The API receives plain strings, and a value never seen in training breaks any encoding built on category codes. Encoding lives inside the sklearn Pipeline (`OneHotEncoder(handle_unknown="ignore")`). When grouping by a category column, pass `observed=` explicitly.
7. **Train and test columns match** apart from `TARGET`.

---
## 2. Data quality

**Question:** Which columns are mostly missing (e.g. the housing block, over 50%)? Which values are placeholders or impossible (`DAYS_EMPLOYED` placeholder, `CODE_GENDER = 'XNA'`, sign of the `DAYS_*` fields)? How extreme are the incomes?

**Decides:** Imputation strategy, missing-indicator features, and which columns to drop.

---
## 3. Signal

**Question:** How does the default rate change across bins or categories of EXT_SOURCE_1/2/3, age, employment length, credit/income and annuity/income ratios, contract type and income type? (Rates, not counts.)

**Decides:** Stage 5 feature-engineering priorities.

---
## 4. Redundancy

**Question:** Which features are near-duplicates (`_AVG` / `_MODE` / `_MEDI` housing triplets, `AMT_CREDIT` vs `AMT_GOODS_PRICE`)?

**Decides:** Feature pruning for the logistic regression baseline.

---
## 5. Fairness groundwork

**Question:** How large is each group, and what is its default rate, by gender, age band and family status? Which features could act as proxies (region rating, population density)?

**Decides:** Base rates for the Stage 6 fairness audit.

---
## 6. Bureau history (time-boxed, `bureau` only)

**Question:** What share of applicants have bureau records, how many records each, and are any records dated after the application?

**Decides:** The bureau aggregation plan, and confirmation of the leakage rule.

---
## 7. Findings

**Question:** What are the 5–10 most important findings, each with its implication?

**Decides:** The summary carried into the README and Stages 4–6.